# Stage 2: Training YOLOv11s-seg untuk Segmentasi Soft Exudate

**Konfigurasi v3:**
- Model: **YOLOv11s-seg (small)** — kapasitas lebih besar untuk menangkap fitur SE yang halus
  - Referensi: Babenko et al. (2024), Frontiers (2025)
- Pre-trained: COCO weights (`yolo11s-seg.pt`, otomatis di-download)
- Device: MPS (Apple Silicon M1 Pro)
- **imgsz=1280** — resolusi lebih tinggi untuk menangkap SE kecil
- Dataset: 54 train, 27 val + **CLAHE preprocessing** pada green channel
- Class: 1 (soft_exudate)

## 1. Cek Environment

In [13]:
import torch
from ultralytics import YOLO
from pathlib import Path

print(f"PyTorch version : {torch.__version__}")
print(f"MPS available   : {torch.backends.mps.is_available()}")
print(f"Device          : {'mps' if torch.backends.mps.is_available() else 'cpu'}")

# Cek file yang dibutuhkan
data_path = Path("yolo_dataset/data.yaml")
print(f"\nData config     : {data_path} — {'OK' if data_path.exists() else 'TIDAK ADA'}")
print(f"Model weights   : yolo11s-seg.pt (akan otomatis di-download jika belum ada)")

PyTorch version : 2.10.0
MPS available   : True
Device          : mps

Data config     : yolo_dataset/data.yaml — OK
Model weights   : yolo11s-seg.pt (akan otomatis di-download jika belum ada)


## 2. Load Model

In [14]:
model = YOLO("yolo11s-seg.pt")
print(f"Model loaded: {model.model_name}")
print(f"Task: {model.task}")

Model loaded: yolo11s-seg.pt
Task: segment


## 3. Training (v3 — YOLOv11s-seg, imgsz=1024, CLAHE dataset)

**Pelajaran dari v3 sebelumnya:**
- `batch=2 + imgsz=1280` menyebabkan training diverge (collapse ke 0)
- YOLO scaling: `effective_lr = lr0 × (nbs/batch)` → batch=2 membuat LR efektif 32x terlalu tinggi
- Gradient dari batch=2 terlalu noisy untuk training stabil

**Konfigurasi v3 (diperbaiki):**
- `yolo11s-seg` (small) — tetap, kapasitas lebih besar dari nano
- `imgsz=1024` — turun dari 1280, agar batch bisa lebih besar
- `batch=4` — naik dari 2, training lebih stabil
- `lr0=0.0005` — diturunkan untuk model small yang lebih besar
- Dataset sudah di-enhance dengan CLAHE pada green channel

In [15]:
model = YOLO("yolo11s-seg.pt")

results = model.train(
    data="yolo_dataset/data.yaml",
    epochs=300,
    patience=80,
    imgsz=1024,
    batch=4,
    device="mps",

    # === Augmentasi (agresif untuk dataset kecil) ===
    augment=True,
    mosaic=1.0,
    close_mosaic=30,
    mixup=0.15,
    copy_paste=0.15,
    degrees=15.0,
    scale=0.5,
    fliplr=0.5,
    flipud=0.5,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,

    # === Optimizer ===
    optimizer="AdamW",
    lr0=0.0005,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=10,

    # === Lainnya ===
    project="runs",
    name="soft_exudate_seg_v3",
    exist_ok=True,
    save=True,
    save_period=50,
    plots=True,
    verbose=True,
)

Ultralytics 8.4.21 🚀 Python-3.14.0 torch-2.10.0 MPS (Apple M1 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=30, cls=0.5, compile=False, conf=None, copy_paste=0.15, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/data.yaml, degrees=15.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=soft_exudate_seg_v3, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=80, perspective=0

## 4. Cek Hasil Training

In [19]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

run_dir = Path("runs/soft_exudate_seg_v3")
if not run_dir.exists():
    run_dir = Path("runs/segment/runs/soft_exudate_seg_v3")

print(f"Run dir: {run_dir}")

results_img = run_dir / "results.png"
if results_img.exists():
    img = Image.open(results_img)
    plt.figure(figsize=(18, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Training Results (v3 — YOLOv11s-seg, imgsz=1280, CLAHE)")
    plt.show()
else:
    print(f"results.png belum ada di {run_dir}")

cm_img = run_dir / "confusion_matrix.png"
if cm_img.exists():
    img = Image.open(cm_img)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion Matrix")
    plt.show()

Run dir: runs/segment/runs/soft_exudate_seg_v3


<Figure size 1800x800 with 1 Axes>

<Figure size 800x800 with 1 Axes>

## 5. Evaluasi pada Validation Set

In [20]:
from pathlib import Path
from ultralytics import YOLO

best_path = Path("runs/soft_exudate_seg_v3/weights/best.pt")
if not best_path.exists():
    best_path = Path("runs/segment/runs/soft_exudate_seg_v3/weights/best.pt")

print(f"Best model: {best_path}")
best_model = YOLO(str(best_path))

metrics = best_model.val(
    data="yolo_dataset/data.yaml",
    device="mps",
    plots=True,
    verbose=True,
)

print("=" * 50)
print("METRIK EVALUASI v3 (Validation Set)")
print("=" * 50)

print(f"\n[Box Detection]")
print(f"  Precision   : {metrics.box.mp:.4f}")
print(f"  Recall      : {metrics.box.mr:.4f}")
print(f"  mAP50       : {metrics.box.map50:.4f}")
print(f"  mAP50-95    : {metrics.box.map:.4f}")

print(f"\n[Mask Segmentation]")
print(f"  Precision   : {metrics.seg.mp:.4f}")
print(f"  Recall      : {metrics.seg.mr:.4f}")
print(f"  mAP50       : {metrics.seg.map50:.4f}")
print(f"  mAP50-95    : {metrics.seg.map:.4f}")

print(f"\n** Recall = Sensitivity (metrik utama skripsi) **")
print(f"\nPerbandingan:")
print(f"  v1 (nano,  640,  tanpa CLAHE) → Mask Recall: 0.526, mAP50: 0.709")
print(f"  v2 (nano,  1024, tanpa CLAHE) → [cek hasil v2 kamu]")
print(f"  v3 (small, 1280, + CLAHE)     → Mask Recall: {metrics.seg.mr:.3f}, mAP50: {metrics.seg.map50:.3f}")
print(f"  Paper sebelumnya              → Sensitivity: 0.454")

Best model: runs/segment/runs/soft_exudate_seg_v3/weights/best.pt
Ultralytics 8.4.21 🚀 Python-3.14.0 torch-2.10.0 MPS (Apple M1 Pro)
YOLO11s-seg summary (fused): 114 layers, 10,067,203 parameters, 0 gradients, 32.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2165.6±471.2 MB/s, size: 1775.1 KB)
val: Scanning /Users/mac/Documents/Kuliah/Semester 6/Skripsi/yolo_dataset/labels/val.cache... 27 images, 13 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 27/27 1.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.4s/it 4.7s<11.0s
                   all         27         38      0.655      0.763      0.734      0.379      0.655      0.763      0.761      0.414
Speed: 0.4ms preprocess, 73.4ms inference, 0.0ms loss, 44.7ms postprocess per image
Results saved to /Users/mac/Documents/Kuliah/Semester 6/Skripsi/runs/segment/val3
METRIK EVALUASI v3 (Validation Set)

[Box Detect

## 6. Visualisasi Prediksi pada Validation Set

In [21]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

val_img_dir = Path("yolo_dataset/images/val")
val_lbl_dir = Path("yolo_dataset/labels/val")
gt_se_val   = Path("2. All Segmentation Groundtruths/b. Testing Set/4. Soft Exudates")

samples = [
    f.stem for f in sorted(val_lbl_dir.glob("*.txt"))
    if f.stat().st_size > 0
][:6]

best_path = Path("runs/soft_exudate_seg_v3/weights/best.pt")
if not best_path.exists():
    best_path = Path("runs/segment/runs/soft_exudate_seg_v3/weights/best.pt")

best_model = YOLO(str(best_path))

fig, axes = plt.subplots(len(samples), 5, figsize=(30, 5 * len(samples)))

for i, stem in enumerate(samples):
    img_path = val_img_dir / f"{stem}.jpg"
    orig = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    img_h, img_w = orig.shape[:2]

    # === Kolom 1: Input (setelah blackout + CLAHE) ===
    axes[i, 0].imshow(orig)
    axes[i, 0].set_title(f"{stem} — Input (blackout + CLAHE)")
    axes[i, 0].axis("off")

    # === Kolom 2: Ground Truth SE overlay ===
    gt_path = gt_se_val / f"{stem}_SE.tif"
    gt_binary = None
    if gt_path.exists():
        gt_mask = cv2.imread(str(gt_path), cv2.IMREAD_UNCHANGED)
        if gt_mask.ndim == 3:
            gt_mask = gt_mask[:, :, :3].max(axis=2)
        gt_binary = (gt_mask > 10).astype(np.uint8)

        overlay = orig.copy()
        overlay[gt_binary > 0] = [0, 255, 0]
        blended = cv2.addWeighted(orig, 0.7, overlay, 0.3, 0)
        axes[i, 1].imshow(blended)
        n_gt = len(cv2.findContours(gt_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0])
        axes[i, 1].set_title(f"{stem} — Ground Truth ({n_gt} SE)")
    else:
        axes[i, 1].imshow(orig)
        axes[i, 1].set_title(f"{stem} — Ground Truth (tidak ada)")
    axes[i, 1].axis("off")

    # === Kolom 3: Prediksi overlay ===
    result = best_model.predict(str(img_path), device="mps", verbose=False)[0]
    pred_img = result.plot()
    pred_img = cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB)
    axes[i, 2].imshow(pred_img)
    n_det = len(result.boxes) if result.boxes is not None else 0
    axes[i, 2].set_title(f"{stem} — Prediksi ({n_det} deteksi)")
    axes[i, 2].axis("off")

    # === Kolom 4: Ground Truth mask (background hitam) ===
    gt_black = np.zeros((img_h, img_w, 3), dtype=np.uint8)
    if gt_binary is not None:
        gt_black[gt_binary > 0] = [0, 255, 0]
    axes[i, 3].imshow(gt_black)
    axes[i, 3].set_title(f"{stem} — GT Mask")
    axes[i, 3].axis("off")

    # === Kolom 5: Prediksi mask (background hitam) ===
    pred_black = np.zeros((img_h, img_w, 3), dtype=np.uint8)
    if result.masks is not None:
        for mask_data in result.masks.data:
            mask_np = mask_data.cpu().numpy()
            # Resize mask ke ukuran gambar asli (YOLO mask bisa beda ukuran)
            if mask_np.shape != (img_h, img_w):
                mask_np = cv2.resize(mask_np, (img_w, img_h), interpolation=cv2.INTER_LINEAR)
            pred_black[mask_np > 0.5] = [0, 120, 255]  # biru/cyan untuk prediksi
    axes[i, 4].imshow(pred_black)
    axes[i, 4].set_title(f"{stem} — Pred Mask ({n_det} deteksi)")
    axes[i, 4].axis("off")

plt.tight_layout()
plt.savefig("visualisasi_prediksi_v3.png", dpi=100, bbox_inches="tight")
plt.show()
print("Visualisasi disimpan ke visualisasi_prediksi_v3.png")

<Figure size 3000x3000 with 30 Axes>

Visualisasi disimpan ke visualisasi_prediksi_v3.png
